# Advanced 09 — Decision Theory and Risk in Statistical Inference

Practice notebook for the **"Decision Theory and Risk in Statistical Inference"** section of the stats course PDF.

## What you will learn

This notebook recaps the theory from the PDF section, then **verifies it with Python**.
Each part ends with a **"Your turn"** prompt; the full **Problem Set** at the end has
**click-to-reveal solutions**.

## How to use

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — statistics is about *relationships*, not memorized formulas.
3. LaTeX uses \( \) for inline math and \[ \] / $$ $$ for display math (KaTeX-friendly).

Let's begin.


---
## Setup — run this first

NumPy for numerics, SciPy for exact distributions, Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

np.random.seed(42)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", scipy.__version__)


---
## Part 1 — Decision rules, loss, and the risk function

An estimator is a **decision rule** \( \delta \) that maps the data \( X \) to an
action (here, an estimate of \( \theta \)). A **loss function**
\( L(\theta, a) \) measures the cost of action \( a \) when the truth is
\( \theta \).

The **risk** of \( \delta \) at \( \theta \) is the expected loss:

$$
R(\theta, \delta) = E_\theta\!\left[ L\!\left(\theta, \delta(X)\right) \right].
$$

We work with the most common choice — **squared-error loss**
\( L(\theta,a)=(\theta-a)^2 \) — for which
\( R(\theta,\delta) = \mathrm{MSE}_\theta(\delta) =
\mathrm{Bias}^2 + \mathrm{Var} \).

**Setup.** Let \( X_1,\dots,X_n \stackrel{iid}{\sim} N(\theta, 1) \). Compare two
estimators of \( \theta \):

- the sample mean \( \delta_1(X)=\bar X \) (unbiased, risk \( = 1/n \));
- a shrinkage estimator \( \delta_2(X)=c\,\bar X \) for some \( 0 \leq c \leq 1 \)
  (biased, lower variance).

We estimate \( R(\theta,\delta) \) by **Monte Carlo**:
draw many samples from \( N(\theta,1) \), apply the rule, average the squared
error.


In [ ]:
def risk_mc(theta, n, delta, M=200_000, seed=0):
    """Monte Carlo estimate of R(theta, delta) for squared-error loss."""
    rng = np.random.default_rng(seed)
    X = rng.normal(theta, 1.0, size=(M, n))
    est = delta(X)
    return np.mean((est - theta) ** 2, axis=0) if est.ndim > 1 else np.mean((est - theta) ** 2)

def mean_rule(X):
    return X.mean(axis=1)

def shrink_rule(c):
    return lambda X: c * X.mean(axis=1)

n = 10
theta_grid = np.linspace(-4, 4, 41)

# Sample mean risk is constant = 1/n = 0.1
R_mean = risk_mc(0.0, n, mean_rule)
print(f"R(theta, mean) ~ {R_mean:.5f}  (theory 1/n = {1/n:.5f})")

# Shrinkage c=0.5: risk = c^2/n + (1-c)^2 theta^2
for c in [0.0, 0.5, 0.8, 1.0]:
    R = [risk_mc(t, n, shrink_rule(c)) for t in theta_grid]
    plt.plot(theta_grid, R, label=f"shrink c={c}")
plt.axhline(1/n, color="k", ls="--", label="sample mean (theory)")
plt.xlabel(r"$\theta$"); plt.ylabel(r"$R(\theta, \delta)$")
plt.title(f"Risk curves, n={n}")
plt.legend(); plt.show()


---
## Part 2 — Admissibility: where does one estimator dominate?

A rule \( \delta_1 \) **dominates** \( \delta_2 \) if
\( R(\theta,\delta_1) \leq R(\theta,\delta_2) \) for all \( \theta \), with
strict inequality somewhere. A rule is **admissible** if no other rule dominates it.

For the shrinkage rule \( \delta_c(X)=c\bar X \) under squared error:

$$
R(\theta, \delta_c) = \frac{c^2}{n} + (1-c)^2\,\theta^2.
$$

The sample mean corresponds to \( c=1 \) with constant risk \( 1/n \). At
\( \theta=0 \), any \( c<1 \) beats the mean (risk \( c^2/n < 1/n \)), but for
large \( |\theta| \) the bias term \( (1-c)^2\theta^2 \) blows up. So \( \delta_c \)
does **not** dominate the mean — the mean is admissible here.

The code below computes the **crossing point** \( |\theta| \) where
\( R(\theta,\delta_c) = 1/n \): the mean wins beyond it, the shrinkage rule wins
inside it.


In [ ]:
def risk_shrink_theory(theta, c, n):
    return c**2 / n + (1 - c) ** 2 * theta**2

n = 10
for c in [0.5, 0.7, 0.9]:
    # solve c^2/n + (1-c)^2 theta^2 = 1/n  ->  theta^2 = (1/n - c^2/n) / (1-c)^2
    cross2 = (1/n - c**2 / n) / (1 - c) ** 2
    cross = np.sqrt(cross2) if cross2 > 0 else np.inf
    print(f"c={c}: shrinkage wins for |theta| < {cross:.3f}")

# Plot with theoretical curves and verify against MC
theta_grid = np.linspace(-3, 3, 61)
plt.axhline(1/n, color="k", ls="--", label="sample mean 1/n")
for c in [0.5, 0.7, 0.9]:
    Rth = risk_shrink_theory(theta_grid, c, n)
    Rmc = [risk_mc(t, n, shrink_rule(c), M=100_000, seed=1000) for t in theta_grid]
    plt.plot(theta_grid, Rth, label=f"theory c={c}")
    plt.plot(theta_grid, Rmc, "o", ms=3, alpha=0.4, label=f"MC c={c}")
plt.xlabel(r"$\theta$"); plt.ylabel(r"$R(\theta, \delta_c)$")
plt.title("Shrinkage risk: theory vs Monte Carlo")
plt.legend(); plt.show()

print("\nThe sample mean is admissible in dimension 1: no c<1 dominates it everywhere.")


---
## Part 3 — James–Stein: inadmissibility of the MLE in dimension \( d \geq 3 \)

Stein's paradox: when estimating a \( d \)-dimensional normal mean
\( \boldsymbol\theta \) from a single observation
\( X \sim N_d(\boldsymbol\theta, I) \), the MLE \( \hat{\boldsymbol\theta}=X \)
is **inadmissible** for \( d \geq 3 \). The James–Stein estimator

$$
\hat{\boldsymbol\theta}^{JS}
= \left( 1 - \frac{d-2}{\|X\|^2} \right) X
$$

has strictly lower risk than \( X \) for **every** \( \boldsymbol\theta \), under
total squared error \( \|\hat\theta - \boldsymbol\theta\|^2 \).

We simulate the risk of both rules as a function of \( \|\boldsymbol\theta\| \).


In [ ]:
def risk_js_mc(theta_vec, M=300_000, seed=0):
    """MC risk of MLE vs James-Stein for X ~ N_d(theta, I), squared-error loss."""
    rng = np.random.default_rng(seed)
    d = len(theta_vec)
    X = rng.normal(theta_vec, 1.0, size=(M, d))
    R_mle = np.mean(np.sum((X - theta_vec) ** 2, axis=1))
    norm2 = np.sum(X ** 2, axis=1)
    shrink = np.maximum(1 - (d - 2) / norm2, 0.0)   # positive-part JS
    X_js = (shrink[:, None]) * X
    R_js = np.mean(np.sum((X_js - theta_vec) ** 2, axis=1))
    return R_mle, R_js

# Vary ||theta|| along e1; risk depends only on ||theta|| by rotational symmetry.
for d in [1, 3, 5, 10]:
    norms = np.linspace(0, 6, 13)
    Rm, Rjs = [], []
    for nm in norms:
        th = np.zeros(d); th[0] = nm
        rm, rj = risk_js_mc(th, M=80_000, seed=d)
        Rm.append(rm); Rjs.append(rj)
    Rm, Rjs = np.array(Rm), np.array(Rjs)
    plt.plot(norms, Rm, "o-", label=f"MLE d={d}")
    plt.plot(norms, Rjs, "s--", label=f"JS  d={d}")
plt.xlabel(r"$\|\theta\|$"); plt.ylabel("risk")
plt.title("James–Stein vs MLE: risk vs ||theta||")
plt.legend(); plt.show()

# Quantify the improvement at theta = 0 and a generic theta
for d in [3, 5, 10]:
    th = np.zeros(d); th[0] = 2.0
    rm, rj = risk_js_mc(th, M=400_000, seed=7)
    print(f"d={d:2d}  R_MLE={rm:.4f}  R_JS={rj:.4f}  improvement={100*(rm-rj)/rm:.1f}%")


---
## Part 4 — Bayes rule, Bayes risk, and the minimax rule

Given a prior \( \pi \) on \( \Theta \), the **Bayes risk** of \( \delta \) is

$$
r(\pi, \delta) = \int R(\theta, \delta)\, \pi(d\theta).
$$

The **Bayes rule** \( \delta_\pi \) minimizes \( r(\pi, \delta) \). Under
squared-error loss, the Bayes rule is the **posterior mean** \( E[\theta \mid X] \):
the value \( a \) minimizing \( E[(\theta-a)^2 \mid X] \) is the mean of the
posterior.

**Conjugate model.** \( X_1,\dots,X_n \mid \theta \sim N(\theta, 1) \) with prior
\( \theta \sim N(0, \tau^2) \). The posterior is
\( \theta \mid X \sim N\!\left( \mu_n, \sigma_n^2 \right) \) with

$$
\mu_n = \frac{n \bar X}{n + 1/\tau^2}, \qquad
\sigma_n^2 = \frac{1}{n + 1/\tau^2}.
$$

So the Bayes rule is \( \delta_\pi(X) = \mu_n = \frac{n}{n + 1/\tau^2}\, \bar X \)
— a shrinkage estimator with \( c = \frac{n}{n + 1/\tau^2} \). The corresponding
Bayes risk is

$$
r(\pi, \delta_\pi) = \sigma_n^2 = \frac{1}{n + 1/\tau^2}.
$$

For a **normal mean with known variance** the unique **minimax** estimator under
squared loss is the sample mean \( \bar X \) (constant risk \( 1/n \)); it is
the limit of Bayes rules as \( \tau \to \infty \) (flat prior). Below we compute
the Bayes rule numerically and compare its Bayes risk to the minimax risk.


In [ ]:
n = 10
tau2_vals = [0.2, 0.5, 1.0, 2.0, 5.0, 1e6]   # last ~ flat prior -> minimax

def bayes_risk_theory(n, tau2):
    return 1.0 / (n + 1.0 / tau2)

def bayes_risk_mc(n, tau2, M=200_000, seed=0):
    """MC estimate of r(pi, delta_pi) = E[ (delta_pi(X) - theta)^2 ] integrated over prior."""
    rng = np.random.default_rng(seed)
    theta = rng.normal(0.0, np.sqrt(tau2), size=M)
    Xbar = rng.normal(theta, 1.0 / np.sqrt(n))
    c = n / (n + 1.0 / tau2)
    delta = c * Xbar
    return np.mean((delta - theta) ** 2)

minimax_risk = 1.0 / n
print(f"{'tau^2':>10} {'c (shrink)':>12} {'Bayes risk theory':>18} {'Bayes risk MC':>16} {'minimax 1/n':>12}")
for tau2 in tau2_vals:
    c = n / (n + 1.0 / tau2)
    r_th = bayes_risk_theory(n, tau2)
    r_mc = bayes_risk_mc(n, tau2)
    tag = "  <- flat prior (minimax limit)" if tau2 > 1e3 else ""
    print(f"{tau2:>10.3g} {c:>12.5f} {r_th:>18.6f} {r_mc:>16.6f} {minimax_risk:>12.6f}{tag}")

# Show the Bayes rule beats minimax when the prior is informative (tau2 small)
# but pays a price for large theta: plot R(theta, delta_pi) vs theta.
theta_grid = np.linspace(-5, 5, 101)
plt.axhline(minimax_risk, color="k", ls="--", label=f"minimax (mean) 1/n={minimax_risk:.3f}")
for tau2 in [0.2, 0.5, 2.0]:
    c = n / (n + 1.0 / tau2)
    R = c**2 / n + (1 - c) ** 2 * theta_grid**2   # theory
    plt.plot(theta_grid, R, label=f"Bayes rule, tau^2={tau2}, c={c:.2f}")
plt.xlabel(r"$\theta$"); plt.ylabel(r"$R(\theta, \delta_\pi)$")
plt.title("Bayes (shrinkage) rules vs minimax sample mean")
plt.legend(); plt.show()


---
## Your turn

- Derive the crossing point \( |\theta| \) where the shrinkage rule
  \( \delta_c \) ties the sample mean, as a function of \( c \) and \( n \).
  Confirm it with `risk_mc`.
- Recompute the James–Stein improvement at \( \boldsymbol\theta = \mathbf 0 \)
  for \( d = 3, 4, \dots, 10 \). What is the theoretical risk of JS at
  \( \mathbf 0 \)? (Recall \( \|X\|^2 \sim \chi^2_d \).)
- Find the prior variance \( \tau^2 \) for which the Bayes rule has Bayes risk
  equal to half the minimax risk \( 1/n \). Verify the formula by Monte Carlo.
- Repeat Part 1 for **absolute-error loss** \( L(\theta,a)=|\theta-a| \): what is
  the risk of the sample median relative to the mean?


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. Show that for squared-error loss the risk of the shrinkage estimator \( \delta_c(X)=c\bar X \) under \( N(\theta,1) \) sampling with \( n \) observations is \( R(\theta,\delta_c)=c^2/n+(1-c)^2\theta^2 \). Use the formula \( \mathrm{MSE}=\mathrm{Bias}^2+\mathrm{Var} \).

2. For which \( c \in [0,1] \) is the shrinkage estimator \( \delta_c \) dominated by the sample mean \( \bar X \)? Find the \( c \) that minimizes the maximum risk \( \sup_\theta R(\theta,\delta_c) \). Compare your answer with \( c=1 \) (the sample mean) and explain why the sample mean is admissible in one dimension.

3. Simulate the James–Stein estimator in dimension \( d=5 \) with \( \boldsymbol\theta = (2,0,0,0,0) \). Estimate the risk of the MLE and of the positive-part James–Stein estimator with \( M=500{,}000 \) Monte Carlo replicates. Report the relative risk reduction and check it is positive.

4. In the conjugate normal model \( X_i\mid\theta\sim N(\theta,1) \), \( \theta\sim N(0,\tau^2) \), derive the Bayes rule under squared-error loss and show its Bayes risk is \( 1/(n+1/\tau^2) \). Then choose \( \tau^2 \) so the Bayes risk equals \( 1/(2n) \) and verify it by Monte Carlo.

5. Explain why the sample mean \( \bar X \) is the **minimax** estimator for a normal mean under squared-error loss, even though it is inadmissible for \( d\geq 3 \). What is its constant risk, and how is it obtained as the limit of Bayes rules?


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** \( \bar X\sim N(\theta,1/n) \) so \( E[c\bar X]=c\theta \) and
\( \mathrm{Bias}=(c-1)\theta \), giving \( \mathrm{Bias}^2=(1-c)^2\theta^2 \).
Also \( \mathrm{Var}(c\bar X)=c^2/n \). Hence
\( R(\theta,\delta_c)=\mathrm{Bias}^2+\mathrm{Var}=(1-c)^2\theta^2+c^2/n \).

```python
import numpy as np
from scipy import stats
n = 10; c = 0.7; theta = 1.3
rng = np.random.default_rng(0)
M = 500_000
Xbar = rng.normal(theta, 1/np.sqrt(n), size=M)
print("theory:", (1-c)**2 * theta**2 + c**2 / n,
      "MC:", np.mean((c*Xbar - theta)**2))
```

**2.** The sample mean dominates \( \delta_c \) (with \( c<1 \)) if
\( R(\theta,\delta_c)\geq 1/n \) for all \( \theta \) with strict inequality
somewhere. The maximum of \( R(\theta,\delta_c) \) over \( \theta \) is infinite
(the \( \theta^2 \) term blows up), so \( \bar X \) does **not** dominate
\( \delta_c \); neither does \( \delta_c \) dominate \( \bar X \) (it loses for
large \( |\theta| \)). The constant-risk minimizer is \( c=1 \): minimize
\( \sup_\theta R \) — the only finite \( \sup_\theta R \) occurs at \( c=1 \),
where \( R\equiv 1/n \). So the sample mean is admissible (in \( d=1 \)).

**3.**
```python
import numpy as np
d = 5; th = np.array([2.0, 0, 0, 0, 0])
rng = np.random.default_rng(123); M = 500_000
X = rng.normal(th, 1.0, size=(M, d))
R_mle = np.mean(np.sum((X - th) ** 2, axis=1))
shrink = np.maximum(1 - (d - 2) / np.sum(X**2, axis=1), 0)
X_js = shrink[:, None] * X
R_js = np.mean(np.sum((X_js - th) ** 2, axis=1))
print(f"R_MLE={R_mle:.4f}  R_JS={R_js:.4f}  reduction={100*(R_mle-R_js)/R_mle:.2f}%")
```
You should see \( R_{JS}<R_{MLE}=d \), a positive reduction of a few percent.

**4.** Posterior \( \theta\mid\bar X \sim N(\mu_n,\sigma_n^2) \) with
\( \mu_n = \frac{n}{n+1/\tau^2}\bar X \) and
\( \sigma_n^2=\frac{1}{n+1/\tau^2} \). The Bayes rule (posterior mean) is
\( \delta_\pi=\mu_n \) with Bayes risk \( r(\pi,\delta_\pi)=\sigma_n^2 \).
Setting \( 1/(n+1/\tau^2)=1/(2n) \) gives \( 1/\tau^2=n \), i.e. \( \tau^2=1/n \).

```python
import numpy as np
n = 10; tau2 = 1/n
rng = np.random.default_rng(7); M = 1_000_000
theta = rng.normal(0, np.sqrt(tau2), M)
Xbar = rng.normal(theta, 1/np.sqrt(n))
c = n / (n + 1/tau2)
print("theory:", 1/(n + 1/tau2), "MC:", np.mean((c*Xbar - theta)**2))
# both ~ 0.05 = 1/(2n)
```

**5.** \( \bar X \) has constant risk \( 1/n \), so
\( \sup_\theta R(\theta,\bar X)=1/n \). It is a **least-favorable-prior** Bayes
rule: with the flat prior \( \tau\to\infty \), the Bayes rule tends to \( \bar X \)
and its Bayes risk tends to \( 1/n \). A Bayes rule with constant risk is minimax, so
\( \bar X \) is minimax. It is inadmissible for \( d\geq 3 \) (James–Stein dominates
it) yet still minimax — minimaxity only requires no rule with smaller *worst-case*
risk, and James–Stein trades large-\( \|\theta\| \) risk for gains near the
origin, so its \( \sup_\theta R \) is not below \( 1/n \).


</details>
